# Exercise 1 — A real VAE on MNIST ($d = 16$)

Train the specified architecture with $\beta = 1$, log reconstruction and KL
separately, then reconstruct from $\mu$ and sample from the prior.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from vae import VAE, evaluate, train_vae

%matplotlib inline

FIG_DIR = Path("..") / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DEVICE = torch.device("cpu")

## A — Load and split the data

In [ ]:
train_full = datasets.MNIST(DATA_DIR, train=True, download=True,
                            transform=transforms.ToTensor())
test = datasets.MNIST(DATA_DIR, train=False, download=True,
                      transform=transforms.ToTensor())
train, val = random_split(train_full, [50_000, 10_000],
                          generator=torch.Generator().manual_seed(42))
print(f"train={len(train)}  val={len(val)}  test={len(test)}")

## B — Build the model

The class in `vae.py` implements the encoder $784\to 512\to 256$, the $\mu$ /
$\log\sigma^2$ heads, the decoder $d\to 256\to 512\to 784$, reparameterization,
and the hand-written BCE + closed-form KL loss.

In [ ]:
torch.manual_seed(42)
model = VAE(latent_dim=16)
print(model)

## C — Train

In [ ]:
hist = train_vae(model, train, val, epochs=20, batch_size=128, lr=1e-3,
                 device=DEVICE)

test_loader = DataLoader(test, batch_size=256, shuffle=False)
test_recon, test_kl = evaluate(model, test_loader, DEVICE)
print(f"test recon={test_recon:.2f}  kl={test_kl:.2f}  "
      f"negELBO={test_recon + test_kl:.2f}")
print(f"val KL first={hist.val_kl[0]:.2f}  last={hist.val_kl[-1]:.2f}")

In [ ]:
epochs = np.arange(1, len(hist.train_recon) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(epochs, hist.train_recon, label="train")
axes[0].plot(epochs, hist.val_recon, label="validation")
axes[0].set_title("Reconstruction (BCE) vs epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Nats per image")
axes[0].legend()

axes[1].plot(epochs, hist.train_kl, label="train")
axes[1].plot(epochs, hist.val_kl, label="validation")
axes[1].set_title("KL vs epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Nats per image")
axes[1].legend()

fig.suptitle("Figure 1 — Training curves ($d = 16$)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## D — Reconstructions and samples

In [ ]:
model.eval()
found = {}
for i in range(len(test)):
    x, y = test[i]
    y = int(y)
    if y not in found:
        found[y] = x
    if len(found) == 10:
        break

with torch.no_grad():
    originals = torch.stack([found[d].view(-1) for d in range(10)])
    mu, _ = model.encode(originals)
    recons = model.decode(mu)

fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for d in range(10):
    axes[0, d].imshow(found[d].view(28, 28), cmap="gray")
    axes[0, d].axis("off")
    axes[0, d].set_title(str(d), fontsize=9)
    axes[1, d].imshow(recons[d].view(28, 28).numpy(), cmap="gray")
    axes[1, d].axis("off")
axes[0, 0].set_ylabel("input", fontsize=9)
axes[1, 0].set_ylabel(r"decode $\mu$", fontsize=9)
fig.suptitle(r"Figure 2 — Reconstruction from $\mu$ (no sampling)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
torch.manual_seed(0)
with torch.no_grad():
    z = torch.randn(64, model.latent_dim)
    imgs = model.decode(z).view(-1, 28, 28).numpy()

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i], cmap="gray")
    ax.axis("off")
fig.suptitle(r"Figure 3 — $8\times 8$ prior samples (seed 0)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_samples.png", dpi=150, bbox_inches="tight")
plt.show()

torch.save(model.state_dict(), "vae_d16.pt")